In [6]:
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon
from IPython.display import display, Markdown
from data_merger import merge_data

## Load Data

In [7]:
data = merge_data()
data

{'2_0': {'NS': {'lost_drones': 1, 'nb_stars': 84, 'time': 127.3599967956543},
  'S': {'lost_drones': 0,
   'nb_stars': 137,
   'time': 150.31999492645264,
   'Q0': 4,
   'Q1': 4,
   'Q2': 4,
   'Q3': 4,
   'Q4': 2,
   'Q5': 4}},
 '2_1': {'NS': {'lost_drones': 3,
   'nb_stars': 122,
   'time': 52.260000228881836,
   'Q0': 3,
   'Q1': 3,
   'Q2': 3,
   'Q3': 5,
   'Q4': 4,
   'Q5': 4},
  'S': {'lost_drones': 0,
   'nb_stars': 98,
   'time': 49.760000228881836,
   'Q0': 3,
   'Q1': 3,
   'Q2': 4,
   'Q3': 5,
   'Q4': 4,
   'Q5': 4}},
 '2_2': {'NS': {'lost_drones': 7, 'nb_stars': 92, 'time': 47.33999824523926},
  'S': {'lost_drones': 7,
   'nb_stars': 99,
   'time': 50.77999687194824,
   'Q0': 1,
   'Q1': 4,
   'Q2': 3,
   'Q3': 4,
   'Q4': 4,
   'Q5': 4}},
 '2_3': {'NS': {'lost_drones': 2,
   'nb_stars': 97,
   'time': 47.279998779296875,
   'Q0': 3,
   'Q1': 2,
   'Q2': 4,
   'Q3': 3,
   'Q4': 2,
   'Q5': 3},
  'S': {'lost_drones': 0,
   'nb_stars': 67,
   'time': 47.09999752044678,
   '

In [ ]:
data["2_2"]["NS"]["Q0"] = 3


{'lost_drones': 7, 'nb_stars': 92, 'time': 47.33999824523926}

## Result stats (paired design)

All questionnaire items used a 5-point Likert scale (1 to 5), where higher values indicate a more favorable rating for that item (anchors differed per question).


In [8]:
def name(course_id):
    if course_id==2: return "Gap"
    if course_id==3: return "Generic"
    return ""

def mean_se(arr):
    arr = np.asarray(arr, dtype=float)
    mean = arr.mean()
    se = arr.std(ddof=1) if len(arr) > 1 else 0.0
    return mean, se

def paired_table(entries, keys, label_map):
    rows = []
    for key in keys:
        s_vals, ns_vals = [], []
        for pid, entry in entries.items():
            s_val = entry['S'].get(key)
            ns_val = entry['NS'].get(key)
            if s_val is None or ns_val is None:
                continue
            s_vals.append(s_val)
            ns_vals.append(ns_val)
        if not s_vals or not ns_vals:
            continue
        s_arr = np.asarray(s_vals, dtype=float)
        ns_arr = np.asarray(ns_vals, dtype=float)
        diff = s_arr - ns_arr
        s_mean, s_se = mean_se(s_arr)
        ns_mean, ns_se = mean_se(ns_arr)
        diff_mean, diff_se = mean_se(diff)
        t_p = ttest_rel(s_arr, ns_arr).pvalue
        try:
            w_p = wilcoxon(diff).pvalue
        except ValueError:
            w_p = np.nan
        rows.append({
            'measure': label_map.get(key, key),
            'n': len(diff),
            'NS_mean': ns_mean,
            'NS_SE': ns_se,
            'S_mean': s_mean,
            'S_SE': s_se,
            'diff_mean': diff_mean,
            'diff_SE': diff_se,
            'p (paired t)': t_p,
            'p (Wilcoxon)': w_p,
        })

    df = pd.DataFrame(rows)
    print(df)
    if df.empty:
        return df, df

    fmt = lambda m, se: f"{m:.2f} [{se:.2f}]"
    formatted = pd.DataFrame({
        'measure': df['measure'],
        'n': df['n'],
        'No-audio mean±sd': [fmt(m, se) for m, se in zip(df['NS_mean'], df['NS_SE'])],
        'Audio mean±sd': [fmt(m, se) for m, se in zip(df['S_mean'], df['S_SE'])],
        'Paired diff (Audio - No Audio)±sd': [fmt(m, se) for m, se in zip(df['diff_mean'], df['diff_SE'])],
        'p (paired t)': df['p (paired t)'].map(lambda p: f"{p:.4f}" if pd.notnull(p) else 'NA'),
        # 'p (Wilcoxon)': df['p (Wilcoxon)'].map(lambda p: f"{p:.4f}" if pd.notnull(p) else 'NA'),
    })
    return df, formatted

# Group entries by obstacle course (batch)
courses = {}
for pid, entry in data.items():
    batch = int(pid.split('_')[0])
    courses.setdefault(batch, {})[pid] = entry

objective_labels = {
    'time': 'Runtime (s)',
    'nb_stars': 'Stars collected',
    'lost_drones': 'Drones lost',
}
question_labels = {
    'Q1': 'Awareness of other drones',
    'Q2': 'Awareness of obstacles',
    'Q3': 'Ease of avoiding obstacles',
    'Q4': 'Alignment with gaps',
    'Q5': 'Cognitive load',
}

objective_stats = {}
objective_tables = {}
question_stats = {}
question_tables = {}

for course_id, entries in sorted(courses.items()):
    raw_obj, table_obj = paired_table(entries, list(objective_labels.keys()), objective_labels)
    objective_stats[course_id] = raw_obj
    objective_tables[course_id] = table_obj

    raw_q, table_q = paired_table(entries, list(question_labels.keys()), question_labels)
    question_stats[course_id] = raw_q
    question_tables[course_id] = table_q

    display(Markdown(f"### {name(course_id)} Course — objective metrics"))
    display(table_obj)
    display(Markdown(f"### {name(course_id)} Course — questionnaire (paired by participant)"))
    display(table_q)

# Familiarity (Q0) summary across participants
q0_per_participant = []
for pid, entry in data.items():
    vals = [entry[cond].get('Q0') for cond in ('S', 'NS') if entry[cond].get('Q0') is not None]
    if vals:
        q0_per_participant.append(np.mean(vals))
if q0_per_participant:
    q0_mean, q0_se = mean_se(q0_per_participant)
    display(Markdown(f"**Familiarity (Q0) overall:** mean {q0_mean:.2f}±{q0_se:.2f}, n = {len(q0_per_participant)}"))

objective_stats, objective_tables, question_stats, question_tables


           measure  n    NS_mean      NS_SE      S_mean       S_SE  diff_mean  \
0      Runtime (s)  6  66.403332  31.434455   86.409997  59.061211  20.006665   
1  Stars collected  6  98.666667  17.704990  110.833333  43.416203  12.166667   
2      Drones lost  6   2.166667   2.639444    1.666667   2.875181  -0.500000   

     diff_SE  p (paired t)  p (Wilcoxon)  
0  40.535012      0.280715      0.562500  
1  39.836750      0.488069      0.500184  
2   2.073644      0.580456      0.580712  
                      measure  n  NS_mean     NS_SE  S_mean      S_SE  \
0   Awareness of other drones  4     2.50  0.577350    3.00  0.816497   
1      Awareness of obstacles  4     2.75  1.258306    4.25  0.500000   
2  Ease of avoiding obstacles  4     3.50  1.290994    4.25  0.957427   
3         Alignment with gaps  4     2.50  1.290994    4.00  0.816497   
4              Cognitive load  4     4.25  0.957427    3.50  0.577350   

   diff_mean   diff_SE  p (paired t)  p (Wilcoxon)  
0       0.5

/opt/anaconda3/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)
/opt/anaconda3/lib/python3.12/site-packages/scipy/stats/_wilcoxon.py:199: UserWarning: Sample size too small for normal approximation.
  temp = _wilcoxon_iv(x, y, zero_method, correction, alternative, method, axis)


### Gap Course — objective metrics

,measure,n,No-audio mean±sd,Audio mean±sd,Paired diff (Audio - No Audio)±sd,p (paired t)
0,Runtime (s),6,66.40 [31.43],86.41 [59.06],20.01 [40.54],0.2807
1,Stars collected,6,98.67 [17.70],110.83 [43.42],12.17 [39.84],0.4881
2,Drones lost,6,2.17 [2.64],1.67 [2.88],-0.50 [2.07],0.5805


### Gap Course — questionnaire (paired by participant)

...

,measure,n,No-audio mean±sd,Audio mean±sd,Paired diff (Audio - No Audio)±sd,p (paired t)
0,Runtime (s),6,36.70 [12.51],48.58 [10.72],11.88 [16.45],0.1371
1,Stars collected,6,14.67 [2.66],18.33 [3.14],3.67 [2.94],0.0284
2,Drones lost,6,2.00 [2.10],0.17 [0.41],-1.83 [1.83],0.0581


### Generic Course — questionnaire (paired by participant)

,measure,n,No-audio mean±sd,Audio mean±sd,Paired diff (Audio - No Audio)±sd,p (paired t)
0,Awareness of other drones,6,2.17 [0.75],3.50 [0.84],1.33 [1.03],0.0250
1,Awareness of obstacles,6,3.67 [1.03],4.50 [0.84],0.83 [0.98],0.0925
2,Ease of avoiding obstacles,6,2.83 [1.17],3.83 [0.75],1.00 [1.26],0.1106
3,Alignment with gaps,6,2.00 [0.63],3.33 [0.82],1.33 [1.37],0.0624
4,Cognitive load,6,3.67 [1.37],3.83 [0.75],0.17 [1.47],0.7926


**Familiarity (Q0) overall:** mean 3.62±1.33, n = 12

({2:            measure  n    NS_mean      NS_SE      S_mean       S_SE  diff_mean  \
  0      Runtime (s)  6  66.403332  31.434455   86.409997  59.061211  20.006665   
  1  Stars collected  6  98.666667  17.704990  110.833333  43.416203  12.166667   
  2      Drones lost  6   2.166667   2.639444    1.666667   2.875181  -0.500000   
  
       diff_SE  p (paired t)  p (Wilcoxon)  
  0  40.535012      0.280715      0.562500  
  1  39.836750      0.488069      0.500184  
  2   2.073644      0.580456      0.580712  ,
  3:            measure  n    NS_mean      NS_SE     S_mean       S_SE  diff_mean  \
  0      Runtime (s)  6  36.703333  12.513082  48.583331  10.715248  11.879998   
  1  Stars collected  6  14.666667   2.658320  18.333333   3.141125   3.666667   
  2      Drones lost  6   2.000000   2.097618   0.166667   0.408248  -1.833333   
  
       diff_SE  p (paired t)  p (Wilcoxon)  
  0  16.448382      0.137095      0.156250  
  1   2.943920      0.028397      0.042168  
  2   1.8348

## Report-ready statements

- **Course 2 objective:** Audio runs were slower (86.41 s [24.11] vs 66.40 s [12.83]; diff +20.01 s [16.55]; paired t p=0.2807, Wilcoxon p=0.5625). Stars were higher with audio (110.83 [17.72] vs 98.67 [7.23]; diff +12.17 [16.26]; p=0.4881/0.5002). Drones lost were slightly lower with audio (1.67 [1.17] vs 2.17 [1.08]; diff -0.50 [0.85]; p=0.5805/0.5807); none of these differences were statistically reliable.
- **Course 3 objective:** Audio runs were slower (48.58 s [4.37] vs 36.70 s [5.11]; diff +11.88 s [6.72]; p=0.1371/0.1562). Stars increased with audio (18.33 [1.28] vs 14.67 [1.09]; diff +3.67 [1.20]; paired t p=0.0284, Wilcoxon p=0.0422). Drones lost decreased with audio (0.17 [0.17] vs 2.00 [0.86]; diff -1.83 [0.75]; p=0.0581/0.0656), a borderline effect.
- **Course 2 questionnaire (n=4 paired):** Audio was associated with higher obstacle awareness (4.25 [0.25] vs 2.75 [0.63]; diff +1.50 [0.50]; p=0.0577/0.1250). Other items showed smaller or opposite differences (e.g., cognitive load -0.75 [0.75]; p=0.3910/0.2763) and were not significant.
- **Course 3 questionnaire (n=6 paired):** Awareness of other drones improved with audio (3.50 [0.34] vs 2.17 [0.31]; diff +1.33 [0.42]; p=0.0250/0.0394). Alignment with gaps trended higher (3.33 [0.33] vs 2.00 [0.26]; diff +1.33 [0.56]; p=0.0624/0.0339). Other items (obstacle awareness, ease of avoidance, cognitive load) had positive but non-significant differences.
- **Familiarity descriptor:** Participants reported familiarity Q0 mean 3.62 [SE 0.38] on the 1–5 scale.
- **Testing note:** Multiple comparisons make these p-values exploratory; interpret claims cautiously and prefer phrasing such as “audio was associated with…” unless supported by the paired tests above.
